# 检测程序是否能正确运行
# 加载配置

In [1]:
import os
import sys

import numpy as np
import pandas as pd

# 将项目根目录添加到 sys.path 以便导入 src 中的模块
# 假设此 notebook 在 'notebooks' 文件夹下，项目根目录是上一级
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import RAW_AUDIO_DIR, PROCESSED_AUDIO_DIR, TRANSCRIPTS_DIR, TARGET_SAMPLE_RATE, TARGET_CHANNELS
from src.utils.logging_utils import get_logger

logger = get_logger('DataPreprocessingNotebook')

LLM Model: Qwen/Qwen2-7B-Instruct
ASR Model: small
Device: cuda
HuggingFace Cache Home: /usr/local/app/jupyterlab/model/hfhome


In [2]:
# 确保目标目录存在
os.makedirs(RAW_AUDIO_DIR, exist_ok=True)
os.makedirs(PROCESSED_AUDIO_DIR, exist_ok=True)
os.makedirs(TRANSCRIPTS_DIR, exist_ok=True)

logger.info(f"原始音频目录: {os.path.abspath(RAW_AUDIO_DIR)}")
logger.info(f"处理后音频目录: {os.path.abspath(PROCESSED_AUDIO_DIR)}")
logger.info(f"文本记录目录: {os.path.abspath(TRANSCRIPTS_DIR)}")
logger.info(f"目标采样率: {TARGET_SAMPLE_RATE} Hz")
logger.info(f"目标通道数: {TARGET_CHANNELS}")

2025-06-10 03:21:34 - DataPreprocessingNotebook - INFO - 原始音频目录: /usr/local/app/jupyterlab/yanjiu/streamllm/data/raw_audio
2025-06-10 03:21:34 - DataPreprocessingNotebook - INFO - 处理后音频目录: /usr/local/app/jupyterlab/yanjiu/streamllm/data/processed_audio
2025-06-10 03:21:34 - DataPreprocessingNotebook - INFO - 文本记录目录: /usr/local/app/jupyterlab/yanjiu/streamllm/data/transcripts
2025-06-10 03:21:34 - DataPreprocessingNotebook - INFO - 目标采样率: 16000 Hz
2025-06-10 03:21:34 - DataPreprocessingNotebook - INFO - 目标通道数: 1


# ASR优化程序运行测试

In [3]:
sys.path.append('..')

from src.asr.faster_whisper_streamer import FasterWhisperStreamer
from src.config import ASR_MODEL_NAME, DEVICE, VAD_PARAMETERS
import soundfile as sf

logger = get_logger(__name__)

def test_asr_streaming():
    """测试ASR流式处理功能"""
    logger.info("测试ASR流式处理...")
    
    # 初始化ASR模型
    asr_streamer = FasterWhisperStreamer(
        model_size=ASR_MODEL_NAME,
        device=DEVICE,
        vad_parameters=VAD_PARAMETERS
    )
    
    # 获取已处理的音频文件
    processed_dir = os.path.join(PROCESSED_AUDIO_DIR, 'length30')
    if not os.path.exists(processed_dir) or len(os.listdir(processed_dir)) == 0:
        logger.warning("没有找到已处理的length10音频文件，请先执行音频处理步骤")
        return
    
    # 选择第一个音频文件进行测试
    audio_files = [f for f in os.listdir(processed_dir) if f.endswith('.wav')]
    if not audio_files:
        logger.warning("length10目录下没有找到wav文件")
        return
    
    test_audio_path = os.path.join(processed_dir, audio_files[0])
    logger.info(f"使用音频文件进行测试: {test_audio_path}")
    
    # 加载音频
    audio_data, sample_rate = sf.read(test_audio_path)
    
    # 如果是立体声转单声道
    if len(audio_data.shape) > 1 and audio_data.shape[1] > 1:
        audio_data = audio_data.mean(axis=1)
    
    # 模拟流式输入 - 每次传入500ms的数据
    chunk_size = int(sample_rate * 0.5)  # 500ms的数据量
    
    # 初始化ASR流
    asr_streamer.start_stream()
    
    text_buffer = []
    
    logger.info("开始模拟音频流...")
    
    # 逐块处理音频数据
    for i in range(0, len(audio_data), chunk_size):
        chunk = audio_data[i:i + chunk_size]
        
        # 向ASR引擎发送音频块
        asr_streamer.process_audio_chunk(chunk, sample_rate)
        
        # 获取当前的转录结果
        text = asr_streamer.get_current_transcript()
        
        if text and text not in text_buffer:
            text_buffer.append(text)
            logger.info(f"当前转录 [{i/sample_rate:.2f}s]: {text}")
    
    # 完成流式处理
    final_text = asr_streamer.finish_stream()
    logger.info(f"最终转录结果: {final_text}")
    
    return final_text

# 执行测试
test_asr_streaming()

2025-06-10 03:22:01 - __main__ - INFO - 测试ASR流式处理...
Loading ASR model: small on cuda with float16


vocabulary.txt:   0%|          | 0.00/460k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.20M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.37k [00:00<?, ?B/s]

model.bin:   0%|          | 0.00/484M [00:00<?, ?B/s]

ASR model loaded.
2025-06-10 03:22:58 - __main__ - INFO - 使用音频文件进行测试: /usr/local/app/jupyterlab/yanjiu/streamllm/data/processed_audio/length10/0_0_d1.wav


AttributeError: 'FasterWhisperStreamer' object has no attribute 'start_stream'